In [ ]:
!pip install snntorch torch torchvision opencv-python

In [ ]:
import torch
print("GPU Ready:", torch.cuda.is_available())

In [ ]:
# 0. Cài đặt thư viện snnTorch nếu chưa có trong session hiện tại
!pip install snntorch -q

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import snntorch as snn

# ==========================================
# 1. Cấu hình các tham số tiền xử lý
# ==========================================
BATCH_SIZE = 64
IMAGE_SIZE = 32  # Thu nhỏ ảnh để tối ưu tốc độ cho Raspberry Pi
NUM_STEPS = 25   # Số bước thời gian (time-steps) cho mỗi khung hình suy luận

# ==========================================
# 2. Xây dựng Pipeline tiền xử lý không gian (Spatial Transforms)
# ==========================================
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # Resize về 32x32 pixels
    transforms.Grayscale(num_output_channels=1),  # Chuyển về ảnh xám (1 kênh) để giảm tính toán
    transforms.ToTensor(),                        # Chuyển thành PyTorch Tensor, tự động scale về [0.0, 1.0]
    transforms.Normalize((0.1307,), (0.3081,))    # Chuẩn hóa phân phối dữ liệu (Mean, Std)
])

# ==========================================
# 3. Tải dữ liệu và cấu hình DataLoader
# ==========================================
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 4. Hàm mã hóa thời gian (Temporal / Spike Encoding)
# ==========================================
def temporal_rate_encoder(data_tensor, num_steps):
    """
    Chuyển đổi tensor ảnh tĩnh [Batch, 1, H, W] thành chuỗi xung điện 
    qua các bước thời gian [Num_Steps, Batch, 1, H, W]
    """
    encoded_seq = torch.stack([data_tensor] * num_steps, dim=0)
    return encoded_seq

# Kiểm tra dữ liệu sau khi tiền xử lý thành công
for images, labels in train_loader:
    print(f"Kích thước Tensor ảnh gốc (Batch, C, H, W): {images.shape}")
    
    # Mã hóa sang miền thời gian
    encoded_data = temporal_rate_encoder(images, NUM_STEPS)
    print(f"Kích thước Tensor sau khi mã hóa thời gian (Num_Steps, Batch, C, H, W): {encoded_data.shape}")
    break

In [ ]:
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

# 1. Khai báo hàm xấp xỉ đạo hàm (Surrogate Gradient)
spike_grad = surrogate.fast_sigmoid(slope=25)

# 2. Xây dựng kiến trúc SNN
class ImageSNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 32x32 = 1024 pixels
        self.fc1 = nn.Linear(32 * 32, 128)
        self.lif1 = snn.Leaky(beta=0.5, spike_grad=spike_grad)
        self.fc2 = nn.Linear(128, 10) # 10 classes
        self.lif2 = snn.Leaky(beta=0.5, spike_grad=spike_grad)

    def forward(self, x, num_steps):
        # Flatten ảnh đầu vào: [Batch, Channel, Height, Width] -> [Batch, Features]
        x = x.view(x.size(0), -1)
        
        # Khởi tạo điện thế màng
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        
        spk2_rec = []
        
        # Vòng lặp qua các bước thời gian (Time-domain unrolling)
        for step in range(num_steps):
            cur1 = self.fc1(x) # Có thể nhân x với chuỗi xung input nếu dùng rate coding phức tạp
            spk1, mem1 = self.lif1(cur1, mem1)
            
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            spk2_rec.append(spk2)
            
        return torch.stack(spk2_rec, dim=0)

# 3. Khởi tạo mô hình, hàm mất mát và Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = ImageSNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.002)

# 4. Vòng lặp huấn luyện tối thiểu (Training Loop)
num_steps = 25
num_epochs = 2

net.train()
for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(train_loader):
        data, targets = data.to(device), targets.to(device)
        
        optimizer.zero_grad()
        spk_out = net(data, num_steps=num_steps)
        
        # Tổng hợp số lượng xung qua các bước thời gian để tính Loss
        sum_spk_out = torch.sum(spk_out, dim=0)
        loss = loss_fn(sum_spk_out, targets)
        
        loss.backward()
        optimizer.step()
        
        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch}/{num_epochs}], Batch [{batch_idx}], Loss: {loss.item():.4f}")

# Lưu trọng số mô hình sau khi train xong
torch.save(net.state_dict(), "snn_model_weights.pth")

In [ ]:
import numpy as np
import torch

# Giả sử mô hình đã train của bạn tên là 'net'
net.eval()

# Trích xuất và lưu trọng số lớp fc1
np.save("weight_fc1.npy", net.fc1.weight.detach().cpu().numpy())
np.save("bias_fc1.npy", net.fc1.bias.detach().cpu().numpy())

# Trích xuất và lưu trọng số lớp fc2
np.save("weight_fc2.npy", net.fc2.weight.detach().cpu().numpy())
np.save("bias_fc2.npy", net.fc2.bias.detach().cpu().numpy())

print("Đã xuất toàn bộ trọng số sang định dạng .npy thành công!")